# Lesson 03 - Agentic Design Patterns

For dis lesson, we go explore three main design patterns wey you fit use build beta AI agents:

1. **Clear Agent Instructions** — Make sharp, role-defined prompts wey go guide how agent go behave
2. **Structured Output with Pydantic Models** — Make sure say agents dey give predictable, validated data
3. **Single Responsibility Agents** — Design agents wey focus on one thing well-well

We go apply each pattern for **travel destination recommender** matter, build system step-by-step wey fit suggest places, check if e dey available, and handle logistics.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: Clear Agent Instructions

Di most impactful pattern na di simplest one: to write clear, detailed instructions for your agent.

Good instructions dey define:
- **Who** di agent be (persona and tone)
- **What** e suppose do (step-by-step responsibilities)
- **How** e suppose behave (constraints and style)

Below, we go create travel concierge agent wey get clear instructions wey dey shape every response wey e go produce.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Structured Output wit Pydantic Models

Free-form text dey use for conversation, but downstream systems need structured data.  
By pairing **Pydantic models** wit a **tool function**, we fit:

- Define exact schema for the agent output  
- Automatically validate responses  
- Take agent results join application logic wey dey reliable  

We still introduce tool wey dey return destination details make the agent fit base hin recommendations on real data.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Single Responsibility Agents

Complex tasks dey benefit when you split work among plenty focused agents, each get only one responsibility:

- A **Destination Expert** wey sabi places and availability
- A **Logistics Planner** wey dey handle flights, hotels, and itineraries

This one dey mirror the software engineering principle of *separation of concerns* — each agent easy to test, maintain, and improve on im own.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

For dis lesson, we apply three agentic design patterns for travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Define persona, responsibilities, and constraints up front | Consistent, on-brand agent behavior |
| **Structured Output** | Use Pydantic models as the response format | Validated, machine-readable results |
| **Single Responsibility** | Give each agent one focused job | Ezi to test, maintain, and compose |

These patterns dey compose naturally — you fit combine clear instructions with structured output inside single-responsibility agent to build strong, production-ready systems.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even if we try make e correct, make you sabi say automated translation fit get mistakes or wahala inside. Di original document wey dem write for di correct language na im be di main source. If na important info, e better make human expert translate am. We no go take any blame if person no understand well or if wahala show for dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
